# Feature Engineering and Selection

INT 374 - PCOS Prediction Project

Selecting important features using statistical tests and information-based methods.
Then creating the final feature-selected dataset.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, f_oneway
from sklearn.feature_selection import mutual_info_classif
import warnings

warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/PCOS_cleaned.csv')
print(f'Loaded: {df.shape}')

Loaded: (541, 42)


In [2]:
# Feature categorization
target = 'PCOS'
numeric_cols = df.select_dtypes(include='number').columns
binary_features = [col for col in df.columns if df[col].nunique() == 2 and col != target]
continuous_features = [col for col in numeric_cols if df[col].nunique() > 10 and col != target]

print(f'Binary features: {len(binary_features)}')
print(f'Continuous features: {len(continuous_features)}')

Binary features: 8
Continuous features: 27


## Chi-Square Test for Binary Features

In [3]:
# Chi-square test for binary features
chi2_scores = {}

for col in binary_features:
    contingency_table = pd.crosstab(df[col], df[target])
    chi2, p_val, dof, expected = chi2_contingency(contingency_table)
    chi2_scores[col] = chi2

chi2_df = pd.DataFrame(list(chi2_scores.items()), columns=['Feature', 'Chi2_Score'])
chi2_df = chi2_df.sort_values('Chi2_Score', ascending=False)

print('Chi-Square Scores for Binary Features:')
print(chi2_df.to_string(index=False))

Chi-Square Scores for Binary Features:
         Feature  Chi2_Score
  Skin_Darkening  120.251400
     Hair_Growth  114.598989
     Weight_Gain  103.306113
       Fast_Food   74.962823
         Pimples   43.064042
       Hair_Loss   15.437094
Regular_Exercise    1.998165
        Pregnant    0.298969


## ANOVA F-Test for Continuous Features

In [4]:
# ANOVA F-test for continuous features
f_scores = {}

for col in continuous_features:
    pcos_0 = df[df[target] == 0][col].dropna()
    pcos_1 = df[df[target] == 1][col].dropna()
    f_stat, p_val = f_oneway(pcos_0, pcos_1)
    f_scores[col] = f_stat

f_df = pd.DataFrame(list(f_scores.items()), columns=['Feature', 'F_Score'])
f_df = f_df.sort_values('F_Score', ascending=False)

print('ANOVA F-Scores for Continuous Features:')
print(f_df.head(15).to_string(index=False))

ANOVA F-Scores for Continuous Features:
            Feature    F_Score
         Follicle_R 390.835930
         Follicle_L 308.519697
                AMH  40.426840
          Weight_Kg  25.349272
                BMI  22.349579
       Cycle_Length  17.734859
                Age  15.753059
              Waist  15.009457
                Hip  14.581533
Avg_Follicle_Size_L   9.704799
       Marriage_Yrs   6.978480
        Endometrium   6.201034
Avg_Follicle_Size_R   5.193400
         Pulse_Rate   4.582952
         Hemoglobin   4.127010


## Mutual Information

In [5]:
# Calculate mutual information
X = df.drop(target, axis=1)
y = df[target]

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print('Mutual Information Scores:')
print(mi_df.head(15).to_string(index=False))

Mutual Information Scores:
            Feature  MI_Score
         Follicle_R  0.240022
         Follicle_L  0.203634
     Skin_Darkening  0.128854
        Hair_Growth  0.102140
        Weight_Gain  0.087883
           Cycle_RI  0.077484
       Cycle_Length  0.073987
                AMH  0.072741
          Prolactin  0.066326
       FSH_LH_Ratio  0.065332
          Fast_Food  0.049640
            Pimples  0.031728
Avg_Follicle_Size_R  0.028855
          Weight_Kg  0.027656
                FSH  0.027232


## Feature Selection Visualization

In [ ]:
# Normalize scores
chi2_scores_norm = (chi2_df.set_index('Feature')['Chi2_Score'] / chi2_df['Chi2_Score'].max()).to_dict()
f_scores_norm = (f_df.set_index('Feature')['F_Score'] / f_df['F_Score'].max()).to_dict()
mi_scores_norm = (mi_df.set_index('Feature')['MI_Score'] / mi_df['MI_Score'].max()).to_dict()

# Combine scores
all_features = set(list(chi2_scores_norm.keys()) + list(f_scores_norm.keys()) + list(mi_scores_norm.keys()))
combined_scores = {}

for feat in all_features:
    score = 0
    if feat in chi2_scores_norm:
        score += chi2_scores_norm[feat]
    if feat in f_scores_norm:
        score += f_scores_norm[feat]
    if feat in mi_scores_norm:
        score += mi_scores_norm[feat]
    combined_scores[feat] = score

combined_df = pd.DataFrame(list(combined_scores.items()), columns=['Feature', 'Combined_Score'])
combined_df = combined_df.sort_values('Combined_Score', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top_20 = combined_df.head(20)
ax.barh(range(len(top_20)), top_20['Combined_Score'].values, color='teal')
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['Feature'].values, fontsize=9)
ax.set_xlabel('Combined Score')
ax.set_title('Top 20 Features by Combined Score')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/27_combined_feature_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved combined ranking plot')

In [ ]:
# Plot Chi-Square scores
fig, ax = plt.subplots(figsize=(10, 6))
top_chi2 = chi2_df.head(15)
ax.barh(range(len(top_chi2)), top_chi2['Chi2_Score'].values, color='purple')
ax.set_yticks(range(len(top_chi2)))
ax.set_yticklabels(top_chi2['Feature'].values, fontsize=9)
ax.set_xlabel('Chi-Square Score')
ax.set_title('Chi-Square Scores (Binary Features)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/25_chi2_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chi-square plot')

In [ ]:
# Plot ANOVA F-scores
fig, ax = plt.subplots(figsize=(10, 8))
top_f = f_df.head(15)
ax.barh(range(len(top_f)), top_f['F_Score'].values, color='orange')
ax.set_yticks(range(len(top_f)))
ax.set_yticklabels(top_f['Feature'].values, fontsize=9)
ax.set_xlabel('F-Score')
ax.set_title('ANOVA F-Scores (Continuous Features)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/24_anova_fscores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved ANOVA plot')

In [ ]:
# Plot Mutual Information scores
fig, ax = plt.subplots(figsize=(10, 8))
top_mi = mi_df.head(15)
ax.barh(range(len(top_mi)), top_mi['MI_Score'].values, color='green')
ax.set_yticks(range(len(top_mi)))
ax.set_yticklabels(top_mi['Feature'].values, fontsize=9)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information Scores')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/26_mutual_info.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved MI plot')

## Select Top Features

In [ ]:
# Select top 30 features
top_features = combined_df.head(30)['Feature'].tolist()
print(f'Selected top 30 features:')
for i, feat in enumerate(top_features, 1):
    print(f'{i}. {feat}')

In [ ]:
# Create final dataset with selected features
selected_features = top_features + [target]
df_final = df[selected_features].copy()

print(f'\nFinal dataset shape: {df_final.shape}')
print(f'Original shape: {df.shape}')
print(f'Features removed: {df.shape[1] - df_final.shape[1]}')

In [ ]:
# Save final dataset
df_final.to_csv('../data/PCOS_final.csv', index=False)
print('Saved: PCOS_final.csv')

In [ ]:
# Save feature selection summary
with open('../outputs/reports/feature_selection_summary.txt', 'w') as f:
    f.write('Feature Selection Summary\n')
    f.write('=' * 50 + '\n')
    f.write(f'Total features in original dataset: {df.shape[1] - 1}\n')
    f.write(f'Features selected: {len(top_features)}\n')
    f.write(f'Final dataset shape: {df_final.shape}\n\n')
    f.write('Selected Features (in order of importance):\n')
    for i, feat in enumerate(top_features, 1):
        f.write(f'{i}. {feat}\n')

print('Saved: feature_selection_summary.txt')
print('\nFeature selection completed')